In [ ]:
import pandas as pd

import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import base64

from modules.pf_calculations import gini, apply_pf_schedule_to_mps, plot_stacked_gain_loss_sortable
from modules.visualisations import plot_profile_by_category

# CONFIG FILE
path_to_local_data = "../../config/"
notebooks_config_file = "notebooks.yaml"

image_filename = "../../../img/icon_logo-1024x1024.png"  # local file path
encoded_image = base64.b64encode(open(image_filename, 'rb').read()).decode()

logo = dict(
    source=f"data:image/png;base64,{encoded_image}",  # local file embedded as base64
    xref="paper", yref="paper",
    x=0.5, y=0.5,            # position: centre
    sizex=0.6, sizey=0.6,    # adjust size
    xanchor="center",
    yanchor="middle",
    opacity=0.2,             # transparency
    layer="below"            # place below the data layer
)

# # PARAMS
# changeable
month = "sept" # "june" # "jan"
full_filepath_to_load = f"org_1_{month}Week.parquet"
work = pd.read_parquet(f"{full_filepath_to_load}")

In [ ]:
def _prepare_sorted_mp_data(data: pd.DataFrame, feature: str):
    df = data.groupby(by="mp_id").sum(numeric_only=True)[feature]
    df = df.sort_values(ascending=False)

    x = np.arange(1, len(df) + 1)
    y = df.values
    mp_ids = df.index

    return df, x, y, mp_ids

def plot_sorted_mps_single(data: pd.DataFrame, feature: str, show=False):
    df, x, y, mp_ids = _prepare_sorted_mp_data(data, feature)

    fig = go.Figure()

    fig.add_trace(go.Bar(
        x=x,
        y=y,
        hovertext=mp_ids,
        hovertemplate="<b>MP_ID:</b> %{hovertext}<br>Value: %{y}<extra></extra>",
        marker=dict(color='lightblue', line=dict(color='black', width=1)),
        name="Sorted values"
    ))

    fig.update_layout(
        title=f"Sorted {df.name}-Values ({len(df)} MP IDs)",
        template="plotly_white",
        xaxis_title="sorted index",
        yaxis_title=df.name,
        height=450,
        showlegend=False
    )

    if show:
        fig.show()

    return fig

def plot_sorted_mps_comparison(
    data: pd.DataFrame,
    feature: str,
    feature_2: str,
    show=False
):
    df1, x1, y1, mp1 = _prepare_sorted_mp_data(data, feature)
    df2, x2, y2, mp2 = _prepare_sorted_mp_data(data, feature_2)

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(feature, feature_2),
        shared_yaxes=True
    )

    fig.add_trace(go.Bar(
        x=x1,
        y=y1,
        hovertext=mp1,
        hovertemplate="<b>MP_ID:</b> %{hovertext}<br>Value: %{y}<extra></extra>",
        marker=dict(color='lightblue', line=dict(color='black', width=1)),
        name="Data 1"
    ), row=1, col=1)

    fig.add_trace(go.Bar(
        x=x2,
        y=y2,
        hovertext=mp2,
        hovertemplate="<b>MP_ID:</b> %{hovertext}<br>Value: %{y}<extra></extra>",
        marker=dict(color='lightblue', line=dict(color='black', width=1)),
        name="Data 2"
    ), row=1, col=2)

    fig.update_layout(
        title=f"Sorted {feature} vs. {feature_2} comparison",
        template="plotly_white",
        height=500,
        showlegend=False
    )

    fig.update_xaxes(title_text="sorted index")
    fig.update_yaxes(title_text=feature)

    if show:
        fig.show()

    return fig

def plot_distribution_single(data: pd.DataFrame, feature: str, show=False):
    df, _, y, _ = _prepare_sorted_mp_data(data, feature)

    fig = make_subplots(
        rows=2, cols=1,
        vertical_spacing=0.15,
        subplot_titles=(f"Histogram [{feature}]", f"Boxplot [{feature}]")
    )

    fig.add_trace(go.Histogram(
        x=y,
        nbinsx=100,
        marker=dict(color='lightblue', line=dict(color='black', width=1)),
        name="Histogram"
    ), row=1, col=1)

    fig.add_trace(go.Box(
        x=y,
        orientation='h',
        boxpoints='outliers',
        marker=dict(color='black'),
        line=dict(color='black'),
        name="Boxplot"
    ), row=2, col=1)

    fig.update_layout(
        title=f"Distribution of {feature}",
        template="plotly_white",
        height=600,
        showlegend=False
    )

    fig.update_xaxes(title_text=feature, row=1, col=1)
    fig.update_yaxes(title_text="count", row=1, col=1)
    fig.update_xaxes(title_text=feature, row=2, col=1)

    if show:
        fig.show()

    return fig

def plot_distribution_comparison(
    data: pd.DataFrame,
    feature: str,
    feature_2: str,
    show=False
):
    _, _, y1, _ = _prepare_sorted_mp_data(data, feature)
    _, _, y2, _ = _prepare_sorted_mp_data(data, feature_2)

    # ------------------------------------------------------------------
    # 1️⃣ Compute common x-range and binning
    # ------------------------------------------------------------------
    all_values = np.concatenate([y1, y2])

    x_min = np.nanmin(all_values)
    x_max = np.nanmax(all_values) + 5

    nbins = 100
    bin_width = (x_max - x_min) / nbins

    # ------------------------------------------------------------------
    # 2️⃣ Subplots
    # ------------------------------------------------------------------
    fig = make_subplots(
        rows=2,
        cols=2,
        vertical_spacing=0.15,
        subplot_titles=(
            f"Histogram – {feature}",
            f"Histogram – {feature_2}",
            f"Boxplot – {feature}",
            f"Boxplot – {feature_2}"
        ),
        #shared_xaxes=True
    )

    # ------------------------------------------------------------------
    # 3️⃣ Histograms with identical bins
    # ------------------------------------------------------------------
    fig.add_trace(go.Histogram(
        x=y1,
        xbins=dict(start=x_min, end=x_max, size=bin_width),
        marker=dict(color='lightblue', line=dict(color='black', width=1)),
        name=feature
    ), row=1, col=1)

    fig.add_trace(go.Histogram(
        x=y2,
        xbins=dict(start=x_min, end=x_max, size=bin_width),
        marker=dict(color='lightblue', line=dict(color='black', width=1)),
        name=feature_2
    ), row=1, col=2)

    # ------------------------------------------------------------------
    # 4️⃣ Boxplots (auto-aligned via shared x-axis)
    # ------------------------------------------------------------------
    fig.add_trace(go.Box(
        x=y1,
        orientation='h',
        boxpoints='outliers',
        marker=dict(color='black'),
        line=dict(color='black'),
        name=feature
    ), row=2, col=1)

    fig.add_trace(go.Box(
        x=y2,
        orientation='h',
        boxpoints='outliers',
        marker=dict(color='black'),
        line=dict(color='black'),
        name=feature_2
    ), row=2, col=2)

    # ------------------------------------------------------------------
    # 5️⃣ Layout & axis settings
    # ------------------------------------------------------------------
    fig.update_layout(
        title=f"Distribution comparison of {feature} vs. {feature_2}",
        template="plotly_white",
        height=700,
        showlegend=False
    )

    # Force identical x-axis range everywhere
    fig.update_xaxes(range=[x_min, x_max], title_text="Value")

    if show:
        fig.show()

    return fig


<!-- ***Optimisation scenario 1***
# Bring large consumers down to the level of households -->

<div>
<img src="enpart_optszenario1.png" width="800"/>
</div>

## Starting point: distribution of the absolute amount of energy that was distributed within the EEG

In [ ]:
#work[work["energy_direction"]=="C"].to_parquet("sorted_original_cons.parquet", index=False)
sorted_original_cons = pd.read_parquet("sorted_original_cons.parquet")
plot_distribution_single(data=sorted_original_cons, feature="comm_cov")

In [ ]:
plot_sorted_mps_single(data=sorted_original_cons, feature="comm_cov")

# Mathematical formalisation

**Decision variable**
- $pf_{z,t,ec}^{\text{*}} \in \mathbb{N}\cap[0,100], \forall{z \in Z}, \forall{t \in T}, \forall{ec \in EC}$ = participation factor for all metering points belonging to those ECs of which the metering point is a member

**Sets**
- $EC$ : energy communities (EC)
- $E$ : feed-in metering points (FMP)
- $V$ : consumption metering points (CMP)
- $Z$ :=  $E \cup V$ (all metering points)
- $T$ : 15-minute time steps
  - $H$ : hours of the day, $H=\{1,\dots ,24\}$  
  - $D$ : days in the planning horizon  

**Parameters (known data)**
- REC membership parameter
  - $m_{z,ec} \in \{0,1\}$, with $m_{z,ec}=1 \leftrightarrow z \text{ is a member of energy community } ec$
- Participation factor at the initial state
  - $pf_{z,t,ec}$ : participation factor
  - Initial state in the current data set: $pf_{z,t,ec} = 100, \forall Z,\forall T,\forall EC$

- For every FMP $e \in E$ and time $t \in T$:
  - $g_{e,t,ec}$ : generation (kWh), how much electricity this metering point generated in total
  - $s_{e,t,ec}$ : surplus (kWh), how much of the generated electricity is, in the context of the REC, surplus and fed into the grid
- For every CMP $v \in V$ and time $t \in T$:
  - $c_{v,t,ec}$ : consumption (kWh), how much electricity this metering point consumed in total
  - $cc_{v,t,ec}$ : community coverage (kWh), how much electricity the metering point could draw from the REC
  - $cp_{v,t,ec}$ : community potential (kWh), how much electricity the metering point could have obtained from the REC. Under under-coverage, this equals the community coverage.

**Water-filling algorithm**
1. Initialisation
   - $a_{v,t}^{(0)} \leftarrow 0, \forall v \in V$ (amount allocated so far)
   - $r_{v,t}^{(0)} \leftarrow c_{v,t}, \forall v \in V$ (full demand)
   - Set $R_t^{(0)} := G_t$ (amount of electricity still to be distributed)
   - Set $U^{(0)} := V$ (CMPs that can still receive electricity / are not yet *saturated*)
2. Iteration $i=0,1,2,\dots$ while $R_t^{(i+1)} > 0$ or $U^{(i+1)} \ne \emptyset$
   1. Fair-share amount for the current round
      - $a_t^{(i)} = \frac{R_t^{(i)}}{|U^{(i)}|}$
   2. Allocation for each still-active consumer
      - $a_{v,t}^{(i+1)} = min(r_{v,t}^{(i)}, a_t^{(i)}), \forall v \in U^{(i)}$
   3. Update the remaining open demand
      - $r_{v,t}^{(i+1)} = r_{v,t}^{(i)} - a_{v,t}^{(i+1)}, \forall v \in U^{(i)}$
   4. Update the remaining amount still to be distributed
      - $R_t^{(i+1)} = R_t^{(i)} - \sum_{v \in U^{i}} a_{v,t}^{(i+1)}$
   5. Update the set of CMPs that can still receive a remainder
      - $U^{(i+1)} = \{v \in U^{(i)} | r_{v,t}^{(i+1)} > 0\}$
   6. Result:
      - $a_{v,t} = \sum_{i \ge 0}a_{v,t}^{(i)}, \forall v \in V$
      - $pf_{v,t}^{\text{*}}  = \frac{a_{v,t}}{c_{v,t}} \cdot 100$

In [ ]:
def plot_sorted_mps(data:pd.DataFrame, feature:str, show=False) -> None:

    df = data.groupby(by="mp_id").sum(numeric_only=True)[feature]

    df = df.sort_values(ascending=False)
    x = np.arange(1, len(df) + 1)
    y = df.values
    mp_ids = df.index  # for hover

    # uniform colours
    bar_color = 'lightblue'
    hist_color = 'lightblue'
    box_color = 'lightblue'
    border_color = 'black'

    # subplots: 3 rows, 1 column
    fig = make_subplots(
        rows=3, cols=1,
        shared_xaxes=False,
        vertical_spacing=0.1,
        subplot_titles=(f"Sorted {df.name}-Values", f"Histogramm [{df.name}]", f"Boxplot [{df.name}]")
    )

    # bar plot on top
    fig.add_trace(go.Bar(
        x=x,
        y=y,
        hovertext=mp_ids,
        hovertemplate="<b>MP_ID:</b> %{hovertext}<br>Value: %{y}<extra></extra>",
        name="Sorted values",
        marker=dict(color=bar_color, line=dict(color=border_color, width=1))
    ), row=1, col=1)

    # histogram in the middle
    fig.add_trace(go.Histogram(
        x=y,
        nbinsx=100,  # adjust number of bins
        name="Histogram",
        marker=dict(color=hist_color, line=dict(color=border_color, width=1)),
    ), row=2, col=1)

    # horizontal boxplot at the bottom
    fig.add_trace(go.Box(
        x=y,
        orientation='h',
        boxpoints='outliers',  # show outliers
        marker=dict(color=border_color),
        line=dict(color=border_color),
        name="Boxplot"
    ), row=3, col=1)

    # layout adjustments
    fig.update_layout(
        title_text=f"{df.name} summed up on single metering points from {start_time.date()} - {end_time.date()} ({len(data.time.unique())} timestamps), {len(df)} mp ids, ",
        template="plotly_white",
        showlegend=False,
        height=900
    )

    # axis titles
    fig.update_xaxes(title_text="sorted Index", row=1, col=1)
    fig.update_yaxes(title_text=df.name, row=1, col=1)
    fig.update_xaxes(title_text=f"{df.name}", row=2, col=1)
    fig.update_yaxes(title_text="count", row=2, col=1)
    fig.update_xaxes(title_text=f"{df.name}", row=3, col=1)
    fig.update_yaxes(title_text="", row=3, col=1)
    
    if show:
        fig.show()
    return fig

In [ ]:
### Waterfilling Opt for finding optimal pfs
mp_counts_on_time = (
    work
    .groupby("time")["energy_direction"]
    .value_counts()
    .unstack(fill_value=0)
    .rename(columns={"C": "count_C_mps", "G": "count_G_mps"})
)
sums_on_time = work.groupby(by="time").sum().reset_index().drop(columns=["mp_id", "energy_direction"]).rename(columns={"wt_meas_cons":"sum_meas_cons", "comm_pot":"sum_comm_pot", "comm_cov":"sum_comm_cov", "wt_meas_gen":"sum_meas_gen", "wt_surp_gen":"sum_surp_gen"})

agg_on_time = pd.merge(left=sums_on_time, right=mp_counts_on_time, on="time", how="outer")

#

time_with_deficit = agg_on_time[agg_on_time["sum_surp_gen"] <= 0]
print(f"{len(time_with_deficit)}/{len(agg_on_time)} ({((len(time_with_deficit)/len(agg_on_time))*100):.4}%) timestamps has deficit. only for consumers during deficit a pf is optimized")
time_with_deficit.head()

#### Acutal optimization calculation


work_tf_cons = pd.merge(left=work[work["energy_direction"] == 'C'], right=time_with_deficit, on="time", how="inner")
# INIT
work_tf_cons["A_0"] = 0
work_tf_cons["a_0"] = 0
work_tf_cons["r_0"] = work_tf_cons["wt_meas_cons"]
work_tf_cons["R_0"] = work_tf_cons["sum_meas_gen"]
work_tf_cons["U_0"] = True # will this VZP still receive generation at this t?
work_tf_cons = work_tf_cons.merge(work_tf_cons.groupby(by="time").sum()["U_0"].rename("U_count_0"), on="time", how="left")

i=0
while (work_tf_cons.groupby(by="time").sum()[f"U_{i}"].rename(f"U_count_{i}").max() > 0) & (work_tf_cons[f"R_{i}"].max() > 0):

    work_tf_cons[f"A_{i}"] = work_tf_cons[f"R_{i}"] / work_tf_cons[f"U_count_{i}"]
    work_tf_cons[f"a_{i+1}"] = work_tf_cons[[f"r_{i}", f"A_{i}"]].min(axis=1)
    work_tf_cons[f"r_{i+1}"] = work_tf_cons[f"r_{i}"] - work_tf_cons[f"a_{i+1}"]

    work_tf_cons = work_tf_cons.merge(work_tf_cons.groupby(by="time").sum()[f"a_{i+1}"].rename(f"sum_a_{i+1}"), on="time", how="left")
    work_tf_cons[f"R_{i+1}"] = work_tf_cons[f"R_{i}"] - work_tf_cons[f"sum_a_{i+1}"]
    work_tf_cons[f"U_{i+1}"] = work_tf_cons[f"r_{i+1}"] > 0
    work_tf_cons = work_tf_cons.merge(work_tf_cons.groupby(by="time").sum()[f"U_{i+1}"].rename(f"U_count_{i+1}"), on="time", how="left")
    i += 1


# sum up for the final distribution
a_cols = [col for col in work_tf_cons.columns if col.startswith("a_")]
work_tf_cons["cc_opt"] = work_tf_cons[a_cols].sum(axis=1).clip(lower=0)
work_tf_cons["pf"] = work_tf_cons["cc_opt"] /  work_tf_cons["wt_meas_cons"] * 100

tf_schedule = work_tf_cons[["time", "mp_id", "pf"]].copy()
tf_schedule["pf"] = tf_schedule["pf"].fillna(100).clip(upper=100)

### Apply pf schedule to energy data
applied_pfs = apply_pf_schedule_to_mps(work, tf_schedule)
applied_pfs["cc_diff"] = applied_pfs["comm_cov"] - applied_pfs["opt_comm_cov"]

#### Summed up for 15min
check_full_calc_via_time = applied_pfs.groupby(by="time").sum()[["wt_meas_cons", "comm_cov", "wt_meas_gen", "opt_meas_cons", "opt_comm_cov"]]

#### Summed up for single MP over whole time horizon
check_full_calcc_via_mpid = applied_pfs.groupby(by="mp_id").sum(numeric_only=True)[["wt_meas_cons", "comm_cov", "wt_meas_gen", "opt_meas_cons", "opt_comm_cov", "cc_diff"]]


# Result of the algorithm: examination of a SINGLE 15min timestamp

In [ ]:
# OPTIMIZATION CHANGES on single timestamp t / single 15min
random_value = np.random.choice(work_tf_cons["time"].unique())
single_time_filtered = applied_pfs[applied_pfs["time"] == random_value]
#### Summed up for single MP over single timestamp
check_full_calc_via_single_timestamp = (
    single_time_filtered[single_time_filtered["energy_direction"] == "C"]
    .groupby(by="mp_id")
    .sum(numeric_only=True)[
        [
            "wt_meas_cons",
            "comm_cov",
            "wt_meas_gen",
            "opt_meas_cons",
            "opt_comm_cov",
            "cc_diff",
        ]
    ]
)

plot_stacked_gain_loss_sortable(
    check_full_calc_via_single_timestamp, "comm_cov", "opt_comm_cov"
)

# Result of the algorithm: looking at the full 7-day period
**15min optimisation does NOT lead to 7-day optimisation**

In [ ]:
# OPTIMIZATION CHANGES on while time horizon
sums_on_cons_mps = applied_pfs[applied_pfs["energy_direction"] == "C"].groupby(by="mp_id").sum(numeric_only=True)
plot_stacked_gain_loss_sortable(sums_on_cons_mps, "comm_cov", "opt_comm_cov")

## Explanation: different load profiles of the consumers

In [ ]:
obj_ids_to_filter = [238, 13]
temp = applied_pfs[applied_pfs["energy_direction"] == "C"]
temp = temp[temp["mp_id"].isin(obj_ids_to_filter)]
plot_profile_by_category(temp, energy_col_name='wt_meas_cons', agg_func_str='median', hue_col="mp_id", logo=logo)

In [ ]:
plot_distribution_comparison(sums_on_cons_mps, "comm_cov", "opt_comm_cov")

In [ ]:
# plot_sorted_mps_comparison(sums_on_cons_mps, "comm_cov", "opt_comm_cov")

# Gini coefficient as the "simplest possible" metric

In [ ]:
# #### Calc Gini Coef over Time
# mps_for_gini_calc = applied_pfs[applied_pfs["energy_direction"] == "C"]

# gini_for_single_timestamp = []
# gini_opt_cc_full_time_horizon = gini(mps_for_gini_calc.groupby(by="mp_id").sum(numeric_only=True)["opt_comm_cov"])
# gini_cc_full_time_horizon = gini(mps_for_gini_calc.groupby(by="mp_id").sum(numeric_only=True)["comm_cov"])
# for act_timestamp in mps_for_gini_calc["time"].unique():
#     filtered_dt = mps_for_gini_calc[mps_for_gini_calc["time"] == act_timestamp]
#     check = filtered_dt[["time", "mp_id", "wt_meas_cons", "comm_cov", "opt_comm_cov", "pf"]]
#     gini_for_single_timestamp.append((act_timestamp, 
#                                       gini(check.groupby(by="mp_id").sum(numeric_only=True)["comm_cov"]), 
#                                       gini(check.groupby(by="mp_id").sum(numeric_only=True)["opt_comm_cov"]),
#                                       filtered_dt.groupby(by="time").sum(numeric_only=True)["comm_cov"].iloc[0]))

# df = pd.DataFrame(
#     gini_for_single_timestamp,
#     columns=["time", "gini_cc", "gini_cc*", "cc_sum"]
# )
# # --- 1) create a complete 15-minute time index ---
# full_range = pd.date_range(
#     start=df["time"].min(),
#     end=df["time"].max(),
#     freq="15min"
# )

# df = df.set_index("time").reindex(full_range)
# df.index.name = "time"
# df.to_parquet("gini_on_timestamps.parquet")


In [ ]:
# Plot Gini over Time
df = pd.read_parquet("gini_on_timestamps.parquet")
fig = make_subplots(specs=[[{"secondary_y": True}]])

mps_for_gini_calc = applied_pfs[applied_pfs["energy_direction"] == "C"]

gini_for_single_timestamp = []
gini_opt_cc_full_time_horizon = gini(mps_for_gini_calc.groupby(by="mp_id").sum(numeric_only=True)["opt_comm_cov"])
gini_cc_full_time_horizon = gini(mps_for_gini_calc.groupby(by="mp_id").sum(numeric_only=True)["comm_cov"])

# colours
color_cc = "#1f77b4"     # blue
color_opt = "#ff7f0e"    # orange
color_sum = "#2ca02c"    # green

# gini_cc
fig.add_trace(go.Scatter(
    x=df.index,
    y=df["gini_cc"],
    mode="lines",
    name="gini_cc",
    connectgaps=False,
    line=dict(color=color_cc)
), secondary_y=False)

# gini_cc*
fig.add_trace(go.Scatter(
    x=df.index,
    y=df["gini_cc*"],
    mode="lines",
    name="gini_cc*",
    connectgaps=False,
    line=dict(color=color_opt)
), secondary_y=False)

# horizontal lines
fig.add_trace(go.Scatter(
    x=[df.index.min(), df.index.max()],
    y=[gini_cc_full_time_horizon, gini_cc_full_time_horizon],
    mode="lines",
    name="gini_cc_full_time_horizon",
    line=dict(color=color_cc, dash="dash")
), secondary_y=False)

fig.add_trace(go.Scatter(
    x=[df.index.min(), df.index.max()],
    y=[gini_opt_cc_full_time_horizon, gini_opt_cc_full_time_horizon],
    mode="lines",
    name="gini_opt_cc_full_time_horizon",
    line=dict(color=color_opt, dash="dash")
), secondary_y=False)

# new line for cc_sum (right axis)
fig.add_trace(go.Scatter(
    x=df.index,
    y=df["cc_sum"],
    mode="lines",
    name="cc_sum",
    line=dict(color=color_sum)
), secondary_y=True)

# layout
fig.update_layout(
    title="Gini Over Time with Fixed Reference Lines and CC Sum",
    xaxis_title="time",
    yaxis_title="gini",
    yaxis=dict(
        title="gini",
        range=[0, 1],        # fixed range
        fixedrange=True     # optional: disable zoom on this axis
    ),
    yaxis2=dict(title="cc_sum", overlaying="y", side="right"),
    template="plotly_white"
)

fig.show()


# Projektstrukturplan
<div>
<img src="Projektstrukturplan_gate2.png" width=700"/>
</div>

## The defined work packages are being implemented **on schedule**
## The project budget is currently being kept.
## There are currently **no project change requests** pending.
<div>
<img src="Projektstrukturplan_gate3.png" width=700"/>
</div>

# Outlook for Gate 4

## Instead of water-filling, set upper limits only for large consumers
## Optimise for longer time horizons
## Instead of optimising at 15min and then converting to hourly participation factors, optimise directly for hourly participation factors